<a href="https://colab.research.google.com/github/desirechiruza/Personal-Projects/blob/main/Copy_of_Option_Pay_off.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install yfinance seaborn requests beautifulsoup4

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import matplotlib.ticker as mticker
import requests
from bs4 import BeautifulSoup

plt.style.use("seaborn-v0_8")

In [3]:
def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

    # Add a User-Agent header to mimic a web browser
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        # Use pandas to read HTML tables from the content of the response
        tables = pd.read_html(response.text)
    except Exception as e:
        raise ValueError(f"Could not read HTML tables from Wikipedia: {e}")

    sp500_df = None
    for table_df in tables:
        # Look for a table that contains a 'Symbol' column
        if 'Symbol' in table_df.columns:
            sp500_df = table_df
            break

    if sp500_df is None:
        raise ValueError("Could not find the S&P 500 constituents table with a 'Symbol' column on Wikipedia.")

    # Extract tickers from the 'Symbol' column
    tickers = sp500_df['Symbol'].tolist()
    # Clean tickers for yfinance compatibility
    tickers = [ticker.strip().replace('.', '-') for ticker in tickers]
    return tickers

In [4]:
def option_payoff_grid(S_range, K, premium, option_type="call", position="long"):
    S = np.array(S_range)

    if option_type == "call":
        payoff = np.maximum(S - K, 0) - premium
    elif option_type == "put":
        payoff = np.maximum(K - S, 0) - premium
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # Adjust payoff for short positions
    if position == "short":
        payoff *= -1

    return S, payoff

In [5]:
def visualize_ticker_options(ticker, n_strikes=5, option_type="call"):
    tk = yf.Ticker(ticker)
    expiries = tk.options
    if not expiries:
        print(f"No options for {ticker}")
        return

    expiry = expiries[0]  # nearest expiry
    if option_type == "call":
        chain = tk.option_chain(expiry).calls
    else:
        chain = tk.option_chain(expiry).puts

    # Drop rows without bid/ask
    chain = chain.dropna(subset=["bid", "ask"])
    if chain.empty:
        print(f"No liquid {option_type}s for {ticker} {expiry}")
        return

    # Approximate underlying spot from lastPrice or ask/bid of ATM
    spot = tk.history(period="1d")["Close"].iloc[-1]

    # Pick strikes around spot
    chain["distance"] = (chain["strike"] - spot).abs()
    chain = chain.sort_values("distance").head(n_strikes)

    S_range = np.linspace(0.5 * spot, 1.5 * spot, 200)

    plt.figure(figsize=(8, 5))
    for _, row in chain.iterrows():
        K = row["strike"]
        mid = (row["bid"] + row["ask"]) / 2.0
        S, payoff = option_payoff_grid(S_range, K, mid, option_type=option_type)
        plt.plot(S, payoff, label=f"K={K:.0f}, mid={mid:.2f}")

    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"{ticker} {option_type.capitalize()} Payoff at Expiry ({expiry})")
    plt.xlabel("Underlying Price at Expiry")
    plt.ylabel("Payoff (per contract, ignoring multiplier)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [6]:
sp500_tickers = get_sp500_tickers()

results = []
S_range = np.linspace(0.5, 1.5, 101)  # relative to spot

# Only run for a small subset to quickly initialize base_spot for the interactive plot
# In a full notebook, you might run this for all tickers or pre-process this data.
for ticker in sp500_tickers[:5]: # Limiting to first 5 tickers for initial setup
    tk = yf.Ticker(ticker)
    expiries = tk.options
    if not expiries:
        continue
    expiry = expiries[0]
    chain = tk.option_chain(expiry).calls.dropna(subset=["bid", "ask"])
    if chain.empty:
        continue

    try:
        spot = tk.history(period="1d")["Close"].iloc[-1]
    except IndexError:
        # Skip ticker if spot price cannot be retrieved
        continue

    chain["distance"] = (chain["strike"] - spot).abs()
    atm_row = chain.sort_values("distance").iloc[0]

    K = atm_row["strike"]
    mid = (atm_row["bid"] + atm_row["ask"]) / 2.0

    S_abs = S_range * spot
    _, payoff = option_payoff_grid(S_abs, K, mid, option_type="call")

    results.append({
        "ticker": ticker,
        "expiry": expiry,
        "spot": spot,
        "strike": K,
        "premium_mid": mid,
        "S_grid": S_abs,
        "payoff_grid": payoff,
    })

results_df = pd.DataFrame(results)

# Initialize base_spot globally, using results_df if available, otherwise a default
if 'results_df' in locals() and not results_df.empty:
    base_spot = results_df['spot'].mean()
else:
    base_spot = 100.0  # Default if results_df is not available or empty

/tmp/ipykernel_65250/1865664910.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [7]:
# Create an Output widget to display hover coordinates
hover_output = widgets.Output()

def interactive_option_payoff_plot(strike, premium, selected_option_type, expiry, spot_range_multiplier=1.0, x_axis_tick_increment=10.0, num_contracts=1, expected_final_price=None):
    global base_spot # Ensure base_spot is accessible

    # Parse the selected_option_type string
    if 'call' in selected_option_type:
        option_kind = 'call'
    elif 'put' in selected_option_type:
        option_kind = 'put'
    else:
        raise ValueError("Invalid option type selected.")

    if 'long' in selected_option_type:
        position = 'long'
    elif 'short' in selected_option_type:
        position = 'short'
    else:
        raise ValueError("Invalid option type selected.")

    # Ensure S_range covers a reasonable spread around the current spot
    S_range = np.linspace(base_spot * (0.5 * spot_range_multiplier), base_spot * (1.5 * spot_range_multiplier), 200)

    S, payoff = option_payoff_grid(S_range, strike, premium, option_type=option_kind, position=position)
    payoff_total = payoff * num_contracts * 100 # Multiply by number of contracts and 100 shares per contract

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(S, payoff_total, label=f'{selected_option_type.replace("_", " ").title()} Payoff (K={strike}, Premium={premium:.2f}, Contracts={num_contracts})')
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(strike, color='grey', linestyle='--', label=f'Strike Price: {strike}')

    # Mark expected final price and its payoff
    if expected_final_price is not None and expected_final_price > 0:
        try:
            # Find the payoff at the expected final price
            # Interpolate if expected_final_price is not exactly in S_range
            expected_payoff = np.interp(expected_final_price, S, payoff_total)
            ax.plot(expected_final_price, expected_payoff, 'ro', markersize=8, label=f'Expected Final Price: {expected_final_price:.2f}, Profit/Loss: {expected_payoff:.2f}')
            ax.axvline(expected_final_price, color='red', linestyle=':', linewidth=0.8)
        except Exception as e:
            print(f"Could not mark expected final price: {e}")

    ax.set_title(f'{selected_option_type.replace("_", " ").title()} Option Payoff Profile for {expiry}')
    ax.set_xlabel('Underlying Price at Expiry')
    ax.set_ylabel('Total Payoff (in $)') # Updated label to reflect total dollars
    ax.grid(True)
    ax.legend()
    plt.tight_layout()

    # Dynamic Y-axis adjustment to emphasize profit area
    min_payoff = payoff_total.min()
    max_payoff = payoff_total.max()
    payoff_range = max_payoff - min_payoff

    if position == 'long':
        # Give more space above zero for long positions (profit is positive)
        ax.set_ylim(min_payoff - 0.1 * payoff_range, max(0, max_payoff) + 0.3 * payoff_range)
    else: # position == 'short'
        # Give more space below zero for short positions (profit is negative)
        ax.set_ylim(min(0, min_payoff) - 0.3 * payoff_range, max_payoff + 0.1 * payoff_range)

    # Ensure the 0 line is always visible
    current_ylim = ax.get_ylim()
    if 0 < current_ylim[0]:
        ax.set_ylim(bottom=0)
    if 0 > current_ylim[1]:
        ax.set_ylim(top=0)

    # Apply x-axis tick increment if provided and positive
    if x_axis_tick_increment > 0:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(x_axis_tick_increment))

    # Function to display coordinates on hover using the Output widget
    def on_motion(event):
        with hover_output:
            hover_output.clear_output(wait=True)
            if event.inaxes == ax:
                print(f"X={event.xdata:.2f}, Y={event.ydata:.2f}")

    # Connect the motion event to the function
    fig.canvas.mpl_connect('motion_notify_event', on_motion)

    plt.show()

# Create interactive widgets
strike_slider = widgets.FloatSlider(
    value=base_spot, # Default strike around the average spot
    min=max(1, base_spot * 0.5), # Ensure min is at least 1
    max=base_spot * 1.5,
    step=1.0,
    description='Strike Price:',
    continuous_update=True
)

premium_slider = widgets.FloatSlider(
    value=base_spot * 0.05, # Default premium, e.g., 5% of spot
    min=0.0,
    max=base_spot * 0.2,
    step=0.1,
    description='Premium:',
    continuous_update=True
)

option_type_selector = widgets.RadioButtons(
    options=['long_call', 'short_call', 'long_put', 'short_put'],
    value='long_call',
    description='Option Type:',
    disabled=False
)

expiry_selector = widgets.Dropdown(
    options=[],
    description='Expiry Date:',
    disabled=False,
)

spot_range_multiplier_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description='X-axis Range Multiplier:',
    continuous_update=True
)

ticker_input = widgets.Text(
    value='AAPL', # Default ticker
    description='Ticker Symbol:',
    disabled=False
)

strike_step_input = widgets.FloatText(
    value=1.0, # Default strike step
    description='Strike Step:',
    disabled=False
)

premium_step_input = widgets.FloatText(
    value=0.1, # Default premium step
    description='Premium Step:',
    disabled=False
)

x_axis_tick_increment_slider = widgets.FloatSlider(
    value=10.0, # Default tick increment
    min=1.0,
    max=100.0,
    step=1.0,
    description='X-axis Tick Increment:',
    continuous_update=True
)

num_contracts_slider = widgets.IntSlider(
    value=1, # Default to 1 contract
    min=1,
    max=100, # Max up to 100 contracts
    step=1,
    description='Number of Contracts:',
    continuous_update=True
)

expected_final_price_input = widgets.FloatText(
    value=base_spot, # Default to current spot
    description='Expected Final Price:',
    disabled=False
)

update_ticker_button = widgets.Button(description="Load Ticker Data")

def update_options_for_expiry(change):
    global base_spot
    ticker_symbol = ticker_input.value.upper()
    selected_expiry = expiry_selector.value
    option_kind = 'call' # Default to call for ATM calculation, will be overridden by option_type_selector

    tk = yf.Ticker(ticker_symbol)

    try:
        spot = tk.history(period="1d")["Close"].iloc[-1]
    except IndexError:
        print(f"Could not retrieve spot price for {ticker_symbol}. Check ticker symbol.")
        return

    # Try to get call chain for ATM strike/premium
    chain = tk.option_chain(selected_expiry).calls.dropna(subset=["bid", "ask"])
    if chain.empty:
        # If no calls, try puts
        chain = tk.option_chain(selected_expiry).puts.dropna(subset=["bid", "ask"])
        if chain.empty:
            print(f"No liquid options for {ticker_symbol} on {selected_expiry}.")
            return

    chain["distance"] = (chain["strike"] - spot).abs()
    atm_row = chain.sort_values("distance").iloc[0]

    atm_strike = atm_row["strike"]
    atm_premium_mid = (atm_row["bid"] + atm_row["ask"]) / 2.0

    base_spot = spot # Update global base_spot for S_range calculation

    # Update slider values
    strike_slider.value = atm_strike
    strike_slider.min = max(1, spot * 0.5) # Adjust min/max around ATM strike for 50% downside
    strike_slider.max = spot * 1.5 # Adjust min/max around ATM strike for 50% upside
    # Use user-defined step if available, otherwise default dynamic step
    strike_slider.step = strike_step_input.value if strike_step_input.value > 0 else (round(spot * 0.01, 1) if spot * 0.01 > 0.1 else 0.1)

    premium_slider.value = atm_premium_mid
    premium_slider.min = 0.0
    premium_slider.max = atm_premium_mid * 5 if atm_premium_mid > 0 else spot * 0.1 # Wider range for premium
    # Use user-defined step if available, otherwise default dynamic step
    premium_slider.step = premium_step_input.value if premium_step_input.value > 0 else (round(atm_premium_mid * 0.05, 2) if atm_premium_mid * 0.05 > 0.01 else 0.01)

    expected_final_price_input.value = spot # Update expected final price to current spot

    print(f"Updated for {ticker_symbol} on {selected_expiry}: Spot={spot:.2f}, ATM Strike={atm_strike:.2f}, Premium={atm_premium_mid:.2f}")

def update_widgets_for_ticker(button):
    global base_spot # Declare base_spot as global to modify it
    ticker_symbol = ticker_input.value.upper() # Ensure uppercase for yfinance
    tk = yf.Ticker(ticker_symbol)

    try:
        spot = tk.history(period="1d")["Close"].iloc[-1]
    except IndexError:
        print(f"Could not retrieve spot price for {ticker_symbol}. Check ticker symbol.")
        return

    expiries = tk.options
    if not expiries:
        print(f"No options available for {ticker_symbol}.")
        return

    # Update expiry selector
    expiry_selector.options = expiries
    expiry_selector.value = expiries[0] # Set default to nearest expiry

    # Call update_options_for_expiry to set initial strike and premium values
    # based on the first expiry. Pass a dummy change object.
    class DummyChange: pass
    update_options_for_expiry(DummyChange())

# Link button to function
update_ticker_button.on_click(update_widgets_for_ticker)

# Link expiry selector to function
expiry_selector.observe(update_options_for_expiry, names='value')

# Initial load of ticker data to set up sliders
# This needs to be called after the widgets are defined but before interactive_plot is linked.
# A dummy button object is passed as the function expects one.
class DummyChange: pass # Re-defining for clarity in this consolidated block
update_widgets_for_ticker(DummyChange())

# Display ticker input and button first
display(ticker_input, update_ticker_button, expiry_selector, strike_step_input, premium_step_input, x_axis_tick_increment_slider, num_contracts_slider, expected_final_price_input)

# Use interact to link the widgets to the plotting function
interactive_plot = widgets.interactive(interactive_option_payoff_plot,
                                       strike=strike_slider,
                                       premium=premium_slider,
                                       selected_option_type=option_type_selector,
                                       expiry=expiry_selector,
                                       spot_range_multiplier=spot_range_multiplier_slider,
                                       x_axis_tick_increment=x_axis_tick_increment_slider,
                                       num_contracts=num_contracts_slider,
                                       expected_final_price=expected_final_price_input)

display(interactive_plot, hover_output)

Updated for AAPL on 2026-03-30: Spot=248.80, ATM Strike=250.00, Premium=1.64
Updated for AAPL on 2026-03-30: Spot=248.80, ATM Strike=250.00, Premium=1.64


Text(value='AAPL', description='Ticker Symbol:')

Button(description='Load Ticker Data', style=ButtonStyle())

Dropdown(description='Expiry Date:', options=('2026-03-30', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04…

FloatText(value=1.0, description='Strike Step:')

...

FloatSlider(value=10.0, description='X-axis Tick Increment:', min=1.0, step=1.0)

IntSlider(value=1, description='Number of Contracts:', min=1)

FloatText(value=248.8000030517578, description='Expected Final Price:')

interactive(children=(FloatSlider(value=250.0, description='Strike Price:', max=373.2000045776367, min=124.400…

Output()